# Step 2a: explaining the profile-stats models

**Models (12):** random forest, XGBoost, LightGBM · RBF SVM, linear SVM · PCA / NMF / SVD + logistic regression · Isolation Forest, LOF, one-class SVM, dense autoencoder.

All of them read the same 12 profile statistics, so they are compared with each other in this notebook (and only with each other; see `docs/step2_plan.md`).

**Two explanation methods per model:** SHAP and LIME. Tree models use the exact TreeSHAP algorithm; every other model uses KernelSHAP, which treats the model as a black box. For the four anomaly detectors, we explain the *anomaly score* (higher = more suspicious).

**What we measure** (all four criteria are computed per explained account, then summarised with a bootstrap 95% interval):

| Criterion | Question | How |
|---|---|---|
| Correctness | Does the explanation find the planted signal? | Rank correlation between |attribution| and the planted per-feature contribution (`truth/accounts_truth.csv`); share of attribution on the 6 truly informative features; top-5 precision |
| Faithfulness | Does deleting the top-attributed features change the score more than deleting random ones? | Replace features by the training mean; effect divided by the std of the model's test scores |
| Agreement | Do SHAP and LIME rank the features alike? | Spearman correlation and top-3 overlap per account |
| Stability | Does a small change to the input leave the explanation alone? | Add noise (5% of each feature's std), re-explain, correlate. A *repeatability* control re-runs the same input with another random seed, so sampling noise in KernelSHAP / LIME can be told apart from real instability |

**Read the results with these known facts in mind** (we chose not to fix them): `posts_per_week` is a noisy copy of `posts_per_day` (credit gets split between them), and the dense autoencoder is a weak model. The PCA and SVD models are mathematically the same model.

> Run the cells top to bottom, one at a time. Each model has its own section, so you can stop and look at the output after each one. The full run takes roughly 15 to 30 minutes on a laptop; set `QUICK = True` for a 2-minute smoke test.

## 0. Setup

In [ ]:
import sys
import time
import warnings
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))  # so `import xai_eval` works from notebooks/

import xai_eval  # noqa: F401  (imports xgboost / lightgbm before torch: macOS OpenMP workaround)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from xai_eval import data as xd
from xai_eval import io as xio
from xai_eval import loaders as L
from xai_eval import metrics as xm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 160)
torch.set_num_threads(4)

import shap
import lime.lime_tabular

In [ ]:
SEED = 0
QUICK = False  # True = tiny samples so the whole notebook runs in a couple of minutes (numbers are then meaningless)

N_PER_KIND = 4 if QUICK else 50      # explained accounts per kind (human / bot / troll) -> 150 accounts in total
N_STAB_PER_KIND = 3 if QUICK else 20 # subset used for the stability / repeatability checks (they cost extra runs)
N_BACKGROUND = 100                   # training accounts used as SHAP background
SHAP_NSAMPLES = 100 if QUICK else 1000   # KernelSHAP: coalitions sampled per explained account
LIME_NSAMPLES = 300 if QUICK else 1500   # LIME: perturbed samples per explained account
KS = [1, 2, 3, 5]                    # deletion sizes for the faithfulness test
NOISE_FRAC = 0.05                    # noise std as a fraction of each feature's std (stability test)
N_RANDOM = 5                         # random deletions averaged per instance
FAMILY = "tabular"

np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Data, the explained sample, and the answer key

In [ ]:
D = xd.load_data(REPO_ROOT)
FEATS = D.FEATURES

# Fixed sample of TEST accounts, the same for every model: equal numbers of human / bot / troll.
S_IDX = D.sample_accounts(per_kind=N_PER_KIND, split="test", seed=SEED)
X_S = D.X_tab[S_IDX]                      # raw feature values of the explained accounts
KIND_S = D.kind[S_IDX]

# Subset for the stability checks (first N_STAB_PER_KIND of each kind, since S_IDX is ordered by kind)
STAB_POS = np.concatenate([np.flatnonzero(KIND_S == k)[:N_STAB_PER_KIND] for k in ("human", "bot", "troll")])

# SHAP background: summarised with k-means so KernelSHAP stays fast
BG_IDX = L.make_background(D, k=N_BACKGROUND, seed=SEED)
BG_SUMMARY = shap.kmeans(D.X_tab[BG_IDX], 10)

# ANSWER KEY (used only to score explanations): per-account planted contribution of each feature
TRUE_C = xd.true_feature_contributions(D, S_IDX)          # (n_sample, 12), 0 for decoys / noise / proxy
INFORMATIVE = np.array([f in xd.INFORMATIVE for f in FEATS])

print("explained accounts:", len(S_IDX), "| stability subset:", len(STAB_POS))
print("truly informative features :", [f for f, m in zip(FEATS, INFORMATIVE) if m])
print("no planted effect          :", [f for f, m in zip(FEATS, INFORMATIVE) if not m])
display(pd.DataFrame({"mean planted |contribution|": TRUE_C.mean(0)}, index=FEATS).round(3).T)

The table above is the answer key: how much each feature truly pushes the log-odds of being inauthentic, on average, for the explained accounts. `burstiness` and `account_age_days` share one interaction term, so both carry its weight. `posts_per_week`, `follower_following_ratio`, `hashtag_rate` and the three noise columns have zero planted effect.

## 2. Reload the step-1 models and check they are the same models

Each model is loaded from `outputs/step1_fit/models/` and asked to score all 5,000 accounts; the result must match the scores step 1 stored. If a row says `MISMATCH`, stop: the explanation would be about a different model.

In [ ]:
MODELS, SCORERS, CHECKS = {}, {}, []
for name in L.TABULAR_MODELS:
    MODELS[name] = L.load_model(name, D)
    SCORERS[name] = L.tabular_scorer(name, MODELS[name], D)   # raw features in, score out
    CHECKS.append(L.verify_reload(name, D, SCORERS[name](D.X_tab)))
assert all(c["status"] == "OK" for c in CHECKS), "A reloaded model does not match its step-1 predictions"

# One scale per model for the faithfulness test: std of its score over the whole TEST split
SCORE_SD = {n: float(np.std(SCORERS[n](D.X_tab[D.is_te]))) for n in L.TABULAR_MODELS}

## 3. Explanation and evaluation code

Everything is defined once here and then reused for every model.

* `explain_shap`: TreeSHAP for the three tree models (exact, no sampling), KernelSHAP for everything else (the model is a black box; we only call its score function).
* `explain_lime`: LIME in regression mode on the model's score.
* `evaluate`: turns the two attribution matrices into the four criteria.

Sign convention: the attribution matrices are `(accounts, 12)` and positive means "pushes the score up (more suspicious)". Correctness, agreement and stability use **|attribution|** (how much the feature matters); the sign is kept in the saved CSVs.

In [ ]:
def _positive_class(sv):
    """SHAP returns different shapes across versions: list per class, or (n, features, classes), or (n, features)."""
    if isinstance(sv, list):
        return np.asarray(sv[-1])
    sv = np.asarray(sv)
    return sv[:, :, -1] if sv.ndim == 3 else sv


def explain_shap(name, X, seed=SEED):
    """Returns (attributions (n, 12), seconds per explained account)."""
    t0 = time.time()
    if name in L.TREE_MODELS:
        # TreeSHAP on the model's own output scale (probability for the forest, log-odds for the boosters).
        ex = shap.TreeExplainer(MODELS[name])
        A = _positive_class(ex.shap_values(pd.DataFrame(X, columns=FEATS)))
    else:
        np.random.seed(seed)  # KernelSHAP samples coalitions with NumPy's global generator
        ex = shap.KernelExplainer(SCORERS[name], BG_SUMMARY)
        A = np.asarray(ex.shap_values(X, nsamples=SHAP_NSAMPLES, silent=True))
    return A, (time.time() - t0) / len(X)


def explain_lime(name, X, seed=SEED):
    t0 = time.time()
    ex = lime.lime_tabular.LimeTabularExplainer(
        D.X_tab[D.is_tr], feature_names=FEATS, mode="regression", random_state=seed
    )
    A = np.zeros((len(X), len(FEATS)))
    for i, x in enumerate(X):
        e = ex.explain_instance(x, SCORERS[name], num_features=len(FEATS), num_samples=LIME_NSAMPLES)
        for j, w in next(iter(e.local_exp.values())):
            A[i, j] = w
    return A, (time.time() - t0) / len(X)


METHODS = {"SHAP": explain_shap, "LIME": explain_lime}

In [ ]:
TRAIN_MEAN = D.X_tab[D.is_tr].mean(0)
FEAT_SD = D.X_tab[D.is_tr].std(0)


def remove_features(x, idx):
    """Neutralise features `idx` of one account by replacing them with the training mean."""
    x = x.copy()
    x[idx] = TRAIN_MEAN[idx]
    return x


def evaluate(name, attrs, secs, scorecard, inst_rows):
    """attrs: {"SHAP": (n,12), "LIME": (n,12)} for the S_IDX sample. Fills the scorecard."""
    f = SCORERS[name]
    supervised = name not in L.TABULAR_ANOMALY
    note_c = "" if supervised else "unsupervised detector: its score was never trained to follow the planted effects"
    predict = lambda rows: f(np.vstack(rows))  # noqa: E731
    X_list = list(X_S)

    for method, A in attrs.items():
        absA = np.abs(A)
        # ---- correctness
        corr = np.array([xm.rank_corr(absA[i], TRUE_C[i]) for i in range(len(A))])
        share = absA[:, INFORMATIVE].sum(1) / np.maximum(absA.sum(1), 1e-12)
        top5 = np.array([xm.top_k_precision(A[i], INFORMATIVE, 5) for i in range(len(A))])
        # ---- faithfulness: deletion test
        dele = xm.deletion_gain(predict, X_list, A, remove_features, KS, SCORE_SD[name], n_random=N_RANDOM, seed=SEED)
        faith = xm.mean_gain_per_instance(dele)
        dele.assign(model=name, method=method).to_csv(xio.out_dir(D, FAMILY) / f"deletion_{name}_{method}.csv", index=False)
        for crit, v, note in [("correctness_rank", corr, note_c), ("correctness_informative_share", share, note_c),
                              ("correctness_top5_precision", top5, note_c), ("faithfulness", faith, "")]:
            scorecard.add(name, method, crit, v, note)
        scorecard.add(name, method, "seconds_per_explanation", secs[method])
        for i in range(len(A)):
            inst_rows.append(dict(model=name, method=method, account_id=int(S_IDX[i]), kind=KIND_S[i],
                                  correctness_rank=corr[i], informative_share=share[i], top5_precision=top5[i],
                                  faithfulness=faith[i]))

    # ---- agreement between the model's two methods
    a, b = np.abs(attrs["SHAP"]), np.abs(attrs["LIME"])
    scorecard.add(name, "SHAP vs LIME", "agreement_spearman", [xm.rank_corr(a[i], b[i]) for i in range(len(a))])
    scorecard.add(name, "SHAP vs LIME", "agreement_top3", [xm.topk_overlap(a[i], b[i], 3) for i in range(len(a))])


def stability_checks(name, attrs, scorecard):
    """Re-explain (i) a slightly noisy copy of the input and (ii) the same input with another random seed."""
    Xs = X_S[STAB_POS]
    Xn = xm.noisy_copy(Xs, FEAT_SD, frac=NOISE_FRAC, seed=SEED + 1)
    for method, fn in METHODS.items():
        base = np.abs(attrs[method][STAB_POS])
        noisy = np.abs(fn(name, Xn, seed=SEED)[0])
        again = np.abs(fn(name, Xs, seed=SEED + 100)[0])  # same input, different random seed
        scorecard.add(name, method, "stability_noise", xm.stability(base, noisy))
        scorecard.add(name, method, "repeatability_seed", xm.stability(base, again),
                      "1.0 expected for TreeSHAP (deterministic)" if (method == "SHAP" and name in L.TREE_MODELS) else "")

In [ ]:
SC = xm.Scorecard("profile_stats")
INST_ROWS, ATTR_STORE = [], {}


def run_model(name, show=True):
    """Explain one model with both methods, evaluate, store, and plot mean |attribution| next to the truth."""
    print(f"=== {name} ===")
    attrs, secs = {}, {}
    for method, fn in METHODS.items():
        attrs[method], secs[method] = fn(name, X_S)
        print(f"  {method}: {secs[method]:.2f} s per account")
    evaluate(name, attrs, secs, SC, INST_ROWS)
    stability_checks(name, attrs, SC)
    ATTR_STORE[name] = attrs
    for method, A in attrs.items():  # signed attributions, for later inspection
        df = pd.DataFrame(A, columns=FEATS)
        df.insert(0, "account_id", S_IDX)
        df.insert(1, "kind", KIND_S)
        xio.save_table(D, FAMILY, f"attributions_{name}_{method}", df)
    if show:
        show_model(name)


def show_model(name):
    a = ATTR_STORE[name]
    fig, ax = plt.subplots(figsize=(9, 3.2))
    w = 0.27
    xs = np.arange(len(FEATS))
    norm = lambda v: v / max(v.sum(), 1e-12)  # noqa: E731
    ax.bar(xs - w, norm(TRUE_C.mean(0)), w, label="truth (planted)", color="#999999")
    ax.bar(xs, norm(np.abs(a["SHAP"]).mean(0)), w, label="SHAP", color="#4C78A8")
    ax.bar(xs + w, norm(np.abs(a["LIME"]).mean(0)), w, label="LIME", color="#F58518")
    ax.set_xticks(xs)
    ax.set_xticklabels(FEATS, rotation=45, ha="right")
    ax.set_ylabel("share of total |attribution|")
    ax.set_title(f"{name}: where does each method put the credit?")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()
    rows = SC.df[(SC.df.model == name) & SC.df.criterion.isin(
        ["correctness_rank", "correctness_informative_share", "faithfulness", "agreement_spearman",
         "stability_noise", "repeatability_seed"])]
    display(rows.pivot_table(index=["method"], columns="criterion", values="value").round(3))

## 4. Tree models: random forest, XGBoost, LightGBM

TreeSHAP is exact and needs no sampling, so any instability in its explanations comes from the model, not from the explainer. LIME is the sampled, model-agnostic counterpart. Trees can represent the planted threshold and interaction directly, so they are the models where a faithful explanation *should* show `burstiness` and `account_age_days` working together.

### 4.1 Random forest

In [ ]:
run_model('rf')

### 4.2 XGBoost

TreeSHAP explains XGBoost's **log-odds** output, not the probability. The ranking of features is what we compare, so this does not affect correctness, agreement or stability; the faithfulness test uses the probability score.

In [ ]:
run_model('xgboost')

### 4.3 LightGBM

In [ ]:
run_model('lightgbm')

## 5. Support vector machines: RBF and linear

The RBF SVM is an opaque kernel model, so both methods are model-agnostic here (KernelSHAP and LIME). The linear SVM is the control: it is a weighted sum of the standardised features, so a correct explanation is easy to state, and any disagreement between methods can only come from the explainers.

### 5.1 RBF SVM

In [ ]:
run_model('svm_rbf')

### 5.2 Linear SVM

In [ ]:
run_model('svm_linear')

## 6. Dimensionality reduction + logistic regression: PCA, NMF, SVD

Each pipeline compresses the 12 features into 6 components, then fits a logistic regression on the components. The explanation is asked about the **original 12 features** (the pipeline is treated as one black box), which is the question an analyst actually has. The components themselves would be a second, harder explanation problem.

PCA and SVD should be identical here (the data are standardised first, so both reduce to the same decomposition); the check after SVD confirms it.

### 6.1 PCA + logistic regression

In [ ]:
run_model('pca_logreg')

### 6.2 NMF + logistic regression

NMF needs non-negative input, so the pipeline rescales every feature to [0, 1] (values outside the training range are clipped). Clipping flattens extreme values, which can make the model insensitive to them and the explanation correspondingly small there.

In [ ]:
run_model('nmf_logreg')

### 6.3 SVD + logistic regression

In [ ]:
run_model('svd_logreg')
# Sanity check: PCA and SVD are the same model, so their explanations should be (nearly) identical
for method in METHODS:
    d = np.abs(ATTR_STORE['pca_logreg'][method] - ATTR_STORE['svd_logreg'][method]).max()
    print(f"max |PCA - SVD| attribution difference, {method}: {d:.2e}")

## 7. Anomaly detectors: Isolation Forest, LOF, one-class SVM, dense autoencoder

These models never saw a label. They score how unusual an account looks, and we hope that inauthentic accounts look unusual. The explanations describe **why the detector finds an account unusual**, which is only partly the same as why the account is fake, so the correctness numbers should be read with that in mind (they are flagged in the scorecard). Faithfulness, agreement and stability are still meaningful.

For the autoencoder the score is the reconstruction error; SHAP and LIME explain that error as a function of the 12 raw features.

### 7.1 Isolation Forest

In [ ]:
run_model('isolation_forest')

### 7.2 Local Outlier Factor

LOF compares an account's local density to its neighbours', so its score depends on the whole neighbourhood, not on a smooth function of the features. Expect noisier explanations.

In [ ]:
run_model('lof')

### 7.3 One-class SVM

In [ ]:
run_model('one_class_svm')

### 7.4 Dense autoencoder

In [ ]:
run_model('autoencoder_dense')

## 8. Scorecard for the profile-stats group

One row per model and method, one column per criterion. Higher is better for every column except `seconds_per_explanation`. There is **no composite score**: with 12 models and several criteria, a single number would hide the trade-offs. Intervals are in `scorecard.csv`.

In [ ]:
tbl = SC.table()
order = ["correctness_rank", "correctness_informative_share", "correctness_top5_precision", "faithfulness",
         "agreement_spearman", "agreement_top3", "stability_noise", "repeatability_seed", "seconds_per_explanation"]
tbl = tbl[[c for c in order if c in tbl.columns]]
display(tbl.round(3))

In [ ]:
# Bar charts with bootstrap intervals, one panel per criterion
df = SC.df
crits = ["correctness_rank", "correctness_informative_share", "faithfulness", "stability_noise"]
fig, axes = plt.subplots(1, len(crits), figsize=(4.2 * len(crits), 6), sharey=True)
models = [m for m in L.TABULAR_MODELS if m in set(df.model)]
for ax, c in zip(axes, crits):
    for k, (method, col) in enumerate([("SHAP", "#4C78A8"), ("LIME", "#F58518")]):
        d = df[(df.criterion == c) & (df.method == method)].set_index("model").reindex(models)
        y = np.arange(len(models)) + (k - 0.5) * 0.38
        ax.barh(y, d["value"], 0.36, color=col, label=method)
        ax.errorbar(d["value"], y, xerr=[(d["value"] - d["ci_low"]).clip(lower=0), (d["ci_high"] - d["value"]).clip(lower=0)],
                    fmt="none", ecolor="black", lw=0.8, capsize=1.5)
    ax.set_title(c.replace("_", " "))
    ax.axvline(0, color="gray", lw=0.8)
axes[0].set_yticks(np.arange(len(models)))
axes[0].set_yticklabels(models)
axes[0].invert_yaxis()
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()

### Reading the scorecard

Use the table in `docs/step2_plan.md` section 2:

* **High faithfulness, low correctness**: the explainer is describing the model accurately, but the model is not using the planted signal. Check the next cell for where the credit went (proxy, decoys or noise).
* **High agreement, low correctness**: both methods share a blind spot (typically crediting `posts_per_week`).
* **Low `stability_noise` but high `repeatability_seed`**: the explanation really changes under a small input change (a model property).
* **Low `repeatability_seed`**: the explainer itself is noisy (a sampling property of KernelSHAP / LIME).

In [ ]:
# Where does each method put its credit? Share of total |attribution| on groups of features.
GROUPS = {
    "informative (6)": [f for f in FEATS if f in xd.INFORMATIVE],
    "proxy posts_per_week": ["posts_per_week"],
    "decoys (2)": ["follower_following_ratio", "hashtag_rate"],
    "noise (3)": ["noise_1", "noise_2", "noise_3"],
}
rows = []
for name, a in ATTR_STORE.items():
    for method, A in a.items():
        absA = np.abs(A)
        tot = absA.sum(1)
        rows.append({"model": name, "method": method,
                     **{g: float((absA[:, [FEATS.index(f) for f in fs]].sum(1) / tot).mean()) for g, fs in GROUPS.items()}})
share_tbl = pd.DataFrame(rows).set_index(["model", "method"])
display(share_tbl.round(3))
print("Planted truth:", {g: round(float(TRUE_C[:, [FEATS.index(f) for f in fs]].sum() / TRUE_C.sum()), 3) for g, fs in GROUPS.items()})
xio.save_table(D, FAMILY, "credit_by_feature_group", share_tbl.reset_index())

In [ ]:
# Deletion curves: how much does the score move when the top-k features (vs random k) are removed?
fig, axes = plt.subplots(3, 4, figsize=(15, 8), sharex=True)
for ax, name in zip(axes.ravel(), L.TABULAR_MODELS):
    for method, col in [("SHAP", "#4C78A8"), ("LIME", "#F58518")]:
        d = pd.read_csv(xio.out_dir(D, FAMILY) / f"deletion_{name}_{method}.csv").groupby("k")[["top", "random"]].mean()
        ax.plot(d.index, d["top"], "-o", color=col, label=f"{method} top-k")
    ax.plot(d.index, d["random"], "--", color="gray", label="random k")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("features deleted")
axes[0, 0].legend(frameon=False, fontsize=8)
fig.supylabel("score change (in score std)")
plt.tight_layout()
plt.show()

In [ ]:
xio.save_scorecard(D, FAMILY, SC, reload_checks=CHECKS, meta=dict(
    seed=SEED, quick=QUICK, n_per_kind=N_PER_KIND, n_background=N_BACKGROUND, shap_nsamples=SHAP_NSAMPLES,
    lime_nsamples=LIME_NSAMPLES, ks=KS, noise_frac=NOISE_FRAC, n_random=N_RANDOM,
    libraries={"shap": shap.__version__}))
pd.DataFrame(INST_ROWS).to_csv(xio.out_dir(D, FAMILY) / "per_instance_scores.csv", index=False)
tbl.round(4).to_csv(xio.out_dir(D, FAMILY) / "scorecard_wide.csv")

## 9. What to take from this notebook

Fill in after reviewing the scorecard (these are the questions the results should answer):

1. Which models give explanations that find the planted features (correctness) *and* describe what the model does (faithfulness)?
2. Do SHAP and LIME agree more for some model types than others? Is disagreement concentrated in the models with interactions or thresholds?
3. Where does the credit for `posts_per_day` go: to itself, to the proxy `posts_per_week`, or split? Does it differ between SHAP and LIME?
4. Are the anomaly detectors explainable in the same sense as the supervised models, or only faithful (they do what they do, but not what the label needs)?
5. Is any explanation unstable enough that it should not be trusted for a single account?

The nonlinear *shape* of the `night_activity_frac` threshold is **not** checked here, because PDP / ALE plots were dropped from the plan; only whether the feature is credited at all.